In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

dataset = dataset.map(
    lambda example: {
        "combined_text": example["prompt"] + " " + example["A"]
    }
)

print(len(dataset[51]["combined_text"]))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sep_token_id = tokenizer.sep_token_id

print(sep_token_id)

102


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

encoded = tokenizer(
    list(dataset["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(dataset[0]["prompt"], return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [7]:
cls_embedding = outputs.last_hidden_state[0, 0]

answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

-1.2001


In [8]:
from transformers import AutoModel
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)
text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

fusion_index = tokens.index("fusion")
print("fusion index:", fusion_index)

attention = outputs.attentions[-1][0, 0]

weight = attention[0, fusion_index].item()

print(f"{weight:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
fusion index: 4
0.1025


In [9]:
import torch
from sentence_transformers import SentenceTransformer, util

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    device = "cuda" if major >= 7 else "cpu"
else:
    device = "cpu"

print(f"Using device: {device}")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(f"{similarity.item():.4f}")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


Using device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np
import torch

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    device = "cuda" if major >= 7 else "cpu"
else:
    device = "cpu"

vectorizer = TfidfVectorizer()

tfidf_top3 = []

for row in dataset:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    tfidf = vectorizer.fit_transform(texts)

    similarities = cosine_similarity(tfidf[0], tfidf[1:]).flatten()

    ranking = np.argsort(similarities)[::-1]

    options = np.array(["A", "B", "C", "D", "E"])

    tfidf_top3.append(options[ranking][:3].tolist())

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

minilm_top3 = []

for row in dataset:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    embeddings = model.encode(
        texts,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    similarities = util.cos_sim(
        embeddings[0],
        embeddings[1:]
    ).cpu().numpy().flatten()

    ranking = np.argsort(similarities)[::-1]

    options = np.array(["A", "B", "C", "D", "E"])

    minilm_top3.append(options[ranking][:3].tolist())

def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0.0
    hits = 0

    for i, p in enumerate(predicted):
        if p == actual:
            hits += 1
            score += hits / (i + 1)

    return score

map3 = np.mean([
    apk(row["answer"], pred)
    for row, pred in zip(dataset, minilm_top3)
])

print("MiniLM MAP@3 =", round(map3, 4))

count = 0

for row, tfidf_pred, minilm_pred in zip(
    dataset,
    tfidf_top3,
    minilm_top3
):

    answer = row["answer"]

    if answer not in tfidf_pred and answer in minilm_pred:
        count += 1

print("Improvement Count =", count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3 = 0.4231
Improvement Count = 564


In [11]:
import torch
from transformers import pipeline

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    device = 0 if major >= 7 else -1
else:
    device = -1

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

sequence = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

result = classifier(sequence, candidate_labels)

print(result["labels"][0])
print(f"{result['scores'][0]:.4f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
0.4575


In [12]:
result_multi = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(result["scores"])

sigmoid_sum = sum(result_multi["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax Sum:", softmax_sum)
print("Sigmoid Sum:", sigmoid_sum)
print(f"{difference:.4f}")

Softmax Sum: 1.0
Sigmoid Sum: 0.0005096028799016494
0.9995


In [13]:
import torch
from transformers import pipeline

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    device = 0 if major >= 7 else -1
else:
    device = -1

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=device
)

text = (
    f"Question: {dataset[0]['prompt']}. "
    f"Is the correct answer A: {dataset[0]['A']} "
    f"or B: {dataset[0]['B']}? "
    f"Answer with just the letter A or B."
)

result = generator(
    text,
    max_new_tokens=5
)

print(result[0]["generated_text"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
